#  Google Play Store — Data Cleaning Pipeline
### Section A | Group 13

This notebook transforms the raw dataset into a **zero-null**, analysis-ready CSV.  
Each cell handles one specific problem. Run them in order.

In [12]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PATH = '../data/Google-Playstore-raw.csv'
CLEAN_PATH = '../data/Google-Playstore-Cleaned.csv'

df = pd.read_csv(RAW_PATH)
print(f'Loaded: {len(df):,} rows, {len(df.columns)} columns')
print(f'Total nulls: {df.isnull().sum().sum():,}')

Loaded: 2,312,944 rows, 24 columns
Total nulls: 1,305,751


---
## Step 1: Drop rows with missing App Name (5 rows)

In [13]:
before = len(df)
df.dropna(subset=['App Name'], inplace=True)
after = len(df)
print(f'Dropped {before - after} rows with null App Name')
print(f'Remaining: {after:,} rows')

Dropped 5 rows with null App Name
Remaining: 2,312,939 rows


---
## Step 2: Fix Rating & Rating Count nulls (22,883 each → fill with 0.0)

In [14]:
print(f'Rating nulls BEFORE    : {df["Rating"].isnull().sum():,}')
print(f'Rating Count nulls BEFORE: {df["Rating Count"].isnull().sum():,}')

# These are apps that have never been rated.
# Mean/Median would be wrong here (you can't "average" a non-existent rating).
# We fill with 0.0 = "No Rating"
df['Rating'] = df['Rating'].fillna(0.0)
df['Rating Count'] = df['Rating Count'].fillna(0.0)

print(f'Rating nulls AFTER     : {df["Rating"].isnull().sum()}')
print(f'Rating Count nulls AFTER : {df["Rating Count"].isnull().sum()}')

Rating nulls BEFORE    : 22,883
Rating Count nulls BEFORE: 22,883
Rating nulls AFTER     : 0
Rating Count nulls AFTER : 0


---
## Step 3: Convert Size strings to numeric MB

In [15]:
def convert_size(size_str):
    if pd.isna(size_str) or 'Varies with device' in str(size_str):
        return np.nan
    size_str = str(size_str).upper().replace(',', '')
    if 'M' in size_str: return float(size_str.replace('M', ''))
    elif 'K' in size_str: return float(size_str.replace('K', '')) / 1024
    elif 'G' in size_str: return float(size_str.replace('G', '')) * 1024
    return np.nan

df['Size_MB'] = df['Size'].apply(convert_size)

# Fill NaN with median (best for skewed numeric data)
size_median = df['Size_MB'].median()
print(f'Size_MB Median: {size_median:.2f} MB')
print(f'Size_MB nulls BEFORE fill: {df["Size_MB"].isnull().sum():,}')
df['Size_MB'] = df['Size_MB'].fillna(size_median)
print(f'Size_MB nulls AFTER fill : {df["Size_MB"].isnull().sum()}')
df['Size'] = df['Size'].fillna('Varies with device')

Size_MB Median: 10.00 MB
Size_MB nulls BEFORE fill: 74,973
Size_MB nulls AFTER fill : 0


---
## Step 4: Convert Installs text to numeric

In [16]:
df['Installs_Numeric'] = (
    df['Installs'].str.replace('+', '', regex=False)
                  .str.replace(',', '', regex=False)
                  .apply(pd.to_numeric, errors='coerce')
                  .fillna(0)
)
df['Installs'] = df['Installs'].fillna('0+')
df['Minimum Installs'] = df['Minimum Installs'].fillna(0)

print(f'Installs_Numeric sample: {df["Installs_Numeric"].head(5).tolist()}')
print(f'Installs nulls: {df["Installs"].isnull().sum()}')

Installs_Numeric sample: [10.0, 5000.0, 50.0, 10.0, 100.0]
Installs nulls: 0


---
## Step 5: Fix Currency nulls + normalize 'XXX'

In [17]:
print(f'Currency nulls BEFORE: {df["Currency"].isnull().sum()}')

# Mode = 'USD' (most common currency)
currency_mode = df['Currency'].mode().iloc[0]
print(f'Currency Mode: {currency_mode}')

df['Currency'] = df['Currency'].fillna(currency_mode)
df['Currency'] = df['Currency'].replace('XXX', 'Unknown')

print(f'Currency nulls AFTER : {df["Currency"].isnull().sum()}')

Currency nulls BEFORE: 135
Currency Mode: USD
Currency nulls AFTER : 0


---
## Step 6: Fix Minimum Android, Developer Info, Released

In [18]:
# Minimum Android — use mode
android_mode = df['Minimum Android'].mode().iloc[0]
print(f'Minimum Android Mode: {android_mode}')
df['Minimum Android'] = df['Minimum Android'].fillna('Varies with device')

# Developer fields
df['Developer Id'] = df['Developer Id'].fillna('Unknown')
df['Developer Email'] = df['Developer Email'].fillna('Unknown')
df['Developer Website'] = df['Developer Website'].fillna('Not Available')

# Released date
df['Released'] = df['Released'].fillna('Unknown')

# Privacy Policy
df['Privacy Policy'] = df['Privacy Policy'].fillna('Not Available')

print(f'\nRemaining nulls after this step:')
remaining = df.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else '✅ All columns clean!')

Minimum Android Mode: 4.1 and up

Remaining nulls after this step:
✅ All columns clean!


---
## Step 7: Convert Dates to Datetime

In [19]:
df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')

print(f'Last Updated dtype: {df["Last Updated"].dtype}')
print(f'Date range: {df["Last Updated"].min()} to {df["Last Updated"].max()}')

Last Updated dtype: datetime64[us]
Date range: 2009-02-09 00:00:00 to 2021-06-16 00:00:00


---
## Step 8: Post-Cleaning Statistics — Mean, Median, Mode

In [20]:
from scipy.stats import kurtosis, skew

cleaned_numeric = ['Rating', 'Rating Count', 'Minimum Installs', 'Price', 'Size_MB', 'Installs_Numeric']

for col in cleaned_numeric:
    s = df[col].dropna()
    print(f'\n--- {col} ---')
    print(f'  Mean     : {s.mean():>15,.4f}')
    print(f'  Median   : {s.median():>15,.4f}')
    print(f'  Mode     : {s.mode().iloc[0] if len(s.mode()) > 0 else "N/A":>15}')
    print(f'  Std Dev  : {s.std():>15,.4f}')
    print(f'  Skewness : {s.skew():>15.4f}')
    print(f'  Kurtosis : {kurtosis(s):>15.4f}')


--- Rating ---
  Mean     :          2.1814
  Median   :          2.8000
  Mode     :             0.0
  Std Dev  :          2.1071
  Skewness :          0.0177
  Kurtosis :         -1.8610

--- Rating Count ---
  Mean     :      2,836.5016
  Median   :          6.0000
  Mode     :             0.0
  Std Dev  :    211,110.8703
  Skewness :        427.9488
  Kurtosis :     229503.9303

--- Minimum Installs ---
  Mean     :    183,437.1020
  Median   :        500.0000
  Mode     :           100.0
  Std Dev  : 15,131,105.4589
  Skewness :        351.1318
  Kurtosis :     155119.3696

--- Price ---
  Mean     :          0.1035
  Median   :          0.0000
  Mode     :             0.0
  Std Dev  :          2.6331
  Skewness :         98.8910
  Kurtosis :      12526.6092

--- Size_MB ---
  Mean     :         18.9098
  Median   :         10.0000
  Mode     :            10.0
  Std Dev  :         23.7118
  Skewness :          5.0057
  Kurtosis :         98.5110

--- Installs_Numeric ---
  Mean  

---
## Step 9: Final Null Check & Export

In [21]:
remaining_nulls = df.isnull().sum()
print('==========================================')
print('  FINAL NULL CHECK')
print('==========================================')
if remaining_nulls.sum() == 0:
    print('  ✅ ZERO NULLS — Dataset clean!')
else:
    print('  ⚠️  Still has nulls:')
    print(remaining_nulls[remaining_nulls > 0])

print(f'\nFinal shape: {df.shape}')

# Export
df.to_csv(CLEAN_PATH, index=False)
print(f'\n✅ Saved to: {CLEAN_PATH}')

  FINAL NULL CHECK
  ✅ ZERO NULLS — Dataset clean!

Final shape: (2312939, 26)

✅ Saved to: ../data/Google-Playstore-Cleaned.csv


---
## Step 10: Before vs After Comparison

In [22]:
raw = pd.read_csv(RAW_PATH)

comparison = pd.DataFrame({
    'Column': raw.columns,
    'Raw Nulls': [raw[c].isnull().sum() for c in raw.columns],
    'Raw Null %': [round(raw[c].isnull().sum()/len(raw)*100, 2) for c in raw.columns],
    'Clean Nulls': [df[c].isnull().sum() if c in df.columns else 'N/A' for c in raw.columns],
})

comparison = comparison.sort_values('Raw Nulls', ascending=False)
print('==========================================')
print('  BEFORE vs AFTER COMPARISON')
print('==========================================')
comparison

  BEFORE vs AFTER COMPARISON


,Column,Raw Nulls,Raw Null %,Clean Nulls
14,Developer Website,760835,32.89,0
19,Privacy Policy,420953,18.20,0
16,Released,71053,3.07,0
3,Rating,22883,0.99,0
4,Rating Count,22883,0.99,0
12,Minimum Android,6530,0.28,0
11,Size,196,0.01,0
10,Currency,135,0.01,0
5,Installs,107,0.00,0
6,Minimum Installs,107,0.00,0
